In [21]:
import sys
print(sys.executable)

c:\Users\yoehe\AppData\Local\Programs\Python\Python312\python.exe


In [22]:
from datasets import Dataset

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

import pandas as pd
import os
import shutil

print("RAGAS / LangChain 환경 정상")

RAGAS / LangChain 환경 정상


In [23]:
PDF_PATH = "../data/sample.pdf"

loader = PyPDFLoader(PDF_PATH)
docs = loader.load()

print("문서 페이지 수:", len(docs))
print(docs[0].page_content[:500])

문서 페이지 수: 1
개강: 2026. 3. 11
아이펠 적응하기 모듈5일
1 (2026.03.11)2 (2026.03.12)3 (2026.03.13)4 (2026.03.16)5 (2026.03.17)
모두의연구소교육철학
아이펠 생활 적응하기: Growth Training
커리어 세미나
아이펠교육시스템Part 1아이펠교육 시스템Part 2
ComputationalThinkingSub QUEST B01 모듈 회고설문
파이썬 마스터모듈9일
6 (2026.03.18)7 (2026.03.19)8 (2026.03.20)9 (2026.03.23)10 (2026.03.24)11 (2026.03.25)12 (2026.03.26)13 (2026.03.27)14 (2026.03.30)
자란다파이썬01-02자란다파이썬03-04
Sub QUEST C02 Sub QUEST C03 Sub QUEST C04 Sub QUEST C05
자란다파이썬05-06 자란다파이썬07-08 자란다파이썬09-10
Slow Paper:Generati


In [24]:
from langchain_community.document_loaders import TextLoader

TXT_PATH = "../data/sample.txt"

loader = TextLoader(TXT_PATH, encoding="utf-8")
docs = loader.load()

print("문서 수:", len(docs))
print(docs[0].page_content[:500])

문서 수: 1
RAG는 Retrieval-Augmented Generation의 약자이다.
문서를 벡터 데이터베이스에 저장한 뒤, 질문과 관련된 문단을 검색하고,
검색된 문맥을 LLM에게 전달하여 답변을 생성한다.

RAG 평가에서는 faithfulness, answer relevancy, context precision, context recall 같은 지표를 사용할 수 있다.
faithfulness는 답변이 검색된 문맥에 근거하는지 평가한다.
context precision은 검색된 문맥 중 실제로 유용한 문맥의 비율을 평가한다.
context recall은 필요한 문맥을 충분히 검색했는지 평가한다.


In [ ]:
from pathlib import Path

content = """# AIFFEL 최종 프로젝트 운영 문서 v1.3
작성일: 2026-05-28
문서 목적: RAG 검색, 근거 기반 답변, 충돌 정보 탐지, 일정/정책/성능지표 질의응답 연습용 샘플 문서

---

## 1. 프로젝트 개요

본 문서는 AI 엔지니어링 학습자가 최종 프로젝트를 준비하면서 참고할 수 있는 운영 문서이다.
프로젝트의 기본 목표는 RAG(Retrieval-Augmented Generation)를 활용하여 개인 학습 자료, 오류 기록, 수업 노트, 프로젝트 산출물을 통합 검색하는 AI 학습 비서를 구현하는 것이다.

최종 산출물은 다음 네 가지 조건을 만족해야 한다.

1. 사용자가 업로드한 문서를 벡터 데이터베이스에 저장할 수 있어야 한다.
2. 질문이 들어오면 관련 문단을 검색하고, 검색된 근거를 함께 제시해야 한다.
3. RAGAS 또는 자체 평가 지표를 사용하여 답변 품질을 분석해야 한다.
4. FastAPI 또는 Streamlit을 이용해 간단한 데모 형태로 배포해야 한다.

이 프로젝트는 단순한 챗봇 제작이 아니라, 검색 품질과 답변 신뢰도를 실험하고 개선하는 것을 핵심으로 한다.

---

## 2. 전체 일정

프로젝트는 총 4단계로 진행된다.

### 2.1 1단계: RAG 기본 파이프라인 구축

기간: 2026-05-28 ~ 2026-06-03

주요 작업:
- PDF 및 TXT 문서 로더 구현
- RecursiveCharacterTextSplitter를 이용한 chunk 분할
- OpenAIEmbeddings 또는 HuggingFace Embeddings 적용
- Chroma 벡터DB 저장
- 질문 입력 후 관련 문서 검색
- 검색된 context와 함께 LLM 답변 생성

완료 기준:
- 최소 3개 문서를 로딩할 수 있어야 한다.
- 질문 10개에 대해 검색된 context를 출력해야 한다.
- 답변에 근거 문단 번호 또는 파일명을 포함해야 한다.

### 2.2 2단계: RAG 평가 실험

기간: 2026-06-04 ~ 2026-06-10

주요 작업:
- chunk_size 500, 1000, 1500 비교
- chunk_overlap 50, 100, 200 비교
- retriever k값 3, 5, 7 비교
- RAGAS 지표 산출
- 점수 낮은 질문에 대한 원인 분석

평가 지표:
- faithfulness
- answer_relevancy
- context_precision
- context_recall

완료 기준:
- 최소 3개 이상의 실험 설정을 비교해야 한다.
- 실험별 평균 점수를 표로 정리해야 한다.
- 가장 좋은 설정을 하나 선택하고 이유를 설명해야 한다.

### 2.3 3단계: 배포형 데모 제작

기간: 2026-06-11 ~ 2026-06-20

주요 작업:
- FastAPI 백엔드 구현
- Streamlit 프론트엔드 구현
- 문서 업로드 기능
- 질문 입력 기능
- 답변 및 근거 문단 출력 기능
- 간단한 사용 로그 저장

완료 기준:
- 로컬 환경에서 실행 가능해야 한다.
- README에 실행 방법이 있어야 한다.
- requirements.txt로 의존성 설치가 가능해야 한다.

### 2.4 4단계: 최종 프로젝트 고도화

기간: 2026-06-21 ~ 2026-07-05

주요 작업:
- 오류 해결 기록 데이터 추가
- 강의 노트 데이터 추가
- 모델 응답 캐싱
- 평가 결과 시각화
- GitHub 정리
- 발표 자료 제작

완료 기준:
- 최종 발표에서 문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.

---

## 3. 시스템 구조

시스템은 다음 구성 요소로 이루어진다.

### 3.1 Document Loader

문서 로더는 PDF, TXT, Markdown 파일을 읽어 LangChain Document 형식으로 변환한다.
각 Document에는 다음 metadata가 포함되어야 한다.

- source: 원본 파일명
- page: PDF 페이지 번호 또는 텍스트 섹션 번호
- created_at: 문서 등록일
- doc_type: pdf, txt, markdown 중 하나

주의:
PDF 문서는 페이지 단위로 분리될 수 있으나, 표나 그림이 많은 PDF는 텍스트 추출 품질이 낮을 수 있다.
이미지 기반 PDF는 OCR이 필요하다.

### 3.2 Text Splitter

chunk 분할은 검색 품질에 큰 영향을 준다.
너무 작은 chunk는 문맥을 잃을 수 있고, 너무 큰 chunk는 검색 결과에 불필요한 정보가 섞일 수 있다.

권장 초기값:
- chunk_size: 1000
- chunk_overlap: 100

다만 한국어 문서는 문장 단위 경계가 명확하지 않은 경우가 있으므로, 실험을 통해 조정해야 한다.

### 3.3 Embedding Model

초기 실험에서는 OpenAIEmbeddings를 사용한다.
추후 비용 절감을 위해 다음 모델을 비교한다.

- intfloat/multilingual-e5-small
- intfloat/multilingual-e5-base
- BAAI/bge-m3
- sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2

한국어 문서가 많을 경우 다국어 embedding 모델을 우선 검토한다.

### 3.4 Vector Database

초기 벡터DB는 Chroma를 사용한다.
운영 환경에서는 Qdrant 또는 PostgreSQL pgvector를 고려할 수 있다.

Chroma 저장 경로:
- ./vectorstore/chroma_default
- ./vectorstore/chroma_chunk_500
- ./vectorstore/chroma_chunk_1000
- ./vectorstore/chroma_chunk_1500

주의:
vectorstore 폴더는 GitHub에 업로드하지 않는다.
벡터DB는 재생성 가능하므로, 실험 코드를 저장하는 것이 더 중요하다.

### 3.5 LLM

초기 실험에서는 gpt-4o-mini를 사용한다.
RunPod 실험에서는 vLLM을 이용해 Qwen 또는 Llama 계열 모델을 사용할 수 있다.

후보 모델:
- Qwen2.5-7B-Instruct
- Qwen2.5-14B-Instruct
- Llama-3.1-8B-Instruct
- Mistral-7B-Instruct

단, 로컬 또는 RunPod에서 오픈소스 모델을 사용할 경우 VRAM 제한을 고려해야 한다.

---

## 4. 실험 설계

### 4.1 Chunk Size 실험

동일한 질문 세트에 대해 chunk_size만 변경한다.

| 실험명 | chunk_size | chunk_overlap | k |
|---|---:|---:|---:|
| exp_chunk_500 | 500 | 50 | 3 |
| exp_chunk_1000 | 1000 | 100 | 3 |
| exp_chunk_1500 | 1500 | 150 | 3 |

가설:
- chunk_size 500은 context_precision이 높을 가능성이 있다.
- chunk_size 1500은 context_recall이 높을 수 있지만 precision은 낮아질 수 있다.
- chunk_size 1000은 두 지표 사이에서 균형이 좋을 가능성이 있다.

### 4.2 Retriever k값 실험

동일한 chunk 설정에서 k값만 변경한다.

| 실험명 | chunk_size | chunk_overlap | k |
|---|---:|---:|---:|
| exp_k_3 | 1000 | 100 | 3 |
| exp_k_5 | 1000 | 100 | 5 |
| exp_k_7 | 1000 | 100 | 7 |

가설:
- k가 증가하면 context_recall은 높아질 수 있다.
- k가 너무 크면 불필요한 context가 포함되어 context_precision이 낮아질 수 있다.
- 답변 모델이 긴 context를 제대로 처리하지 못하면 faithfulness가 낮아질 수 있다.

### 4.3 Embedding Model 실험

같은 chunk 설정에서 embedding 모델을 변경한다.

| 실험명 | embedding_model | 예상 특징 |
|---|---|---|
| exp_openai | OpenAIEmbeddings | 안정적이지만 비용 발생 |
| exp_e5_small | multilingual-e5-small | 가볍고 빠르지만 성능 제한 |
| exp_e5_base | multilingual-e5-base | 균형형 |
| exp_bge_m3 | BAAI/bge-m3 | 다국어 검색에 강점 |

---

## 5. 평가 질문 세트

다음 질문은 RAGAS 평가용 기본 질문이다.

Q1. 이 프로젝트의 최종 목표는 무엇인가?
기준 답변: 이 프로젝트의 최종 목표는 RAG 기반 AI 학습 비서를 구현하고, 검색 품질과 답변 신뢰도를 평가 및 개선하는 것이다.

Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?
기준 답변: chunk_size가 너무 작으면 문맥이 잘려 답변에 필요한 정보가 충분히 검색되지 않을 수 있다.

Q3. chunk_size가 너무 크면 어떤 문제가 생기는가?
기준 답변: chunk_size가 너무 크면 검색 결과에 불필요한 정보가 섞여 context_precision이 낮아질 수 있다.

Q4. RAGAS에서 faithfulness는 무엇을 평가하는가?
기준 답변: faithfulness는 답변이 검색된 context에 근거하여 사실적으로 작성되었는지를 평가한다.

Q5. vectorstore 폴더를 GitHub에 올리지 않는 이유는 무엇인가?
기준 답변: vectorstore는 용량이 크고 재생성 가능하기 때문에 GitHub에 올리지 않고, 생성 코드만 관리하는 것이 적절하다.

Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는 이유는 무엇인가?
기준 답변: 최종 프로젝트의 모델과 데이터 규모가 확정되지 않았을 때는 저장공간을 먼저 고정하면 비용과 GPU 선택 제약이 생길 수 있기 때문이다.

Q7. OpenAIEmbeddings를 사용하면 RunPod GPU를 많이 사용하는가?
기준 답변: OpenAIEmbeddings는 OpenAI 서버에서 계산되므로 RunPod GPU 사용량은 크지 않다.

Q8. retriever의 k값을 높이면 항상 좋은가?
기준 답변: 아니다. k값을 높이면 recall은 좋아질 수 있지만 불필요한 문맥이 포함되어 precision이나 faithfulness가 낮아질 수 있다.

Q9. 배포형 데모에서 필요한 핵심 기능은 무엇인가?
기준 답변: 문서 업로드, 질문 입력, 답변 출력, 근거 문단 출력, 간단한 사용 로그 저장 기능이 필요하다.

Q10. 최종 발표에서 반드시 설명해야 할 항목은 무엇인가?
기준 답변: 문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.

---

## 6. 운영 규칙

### 6.1 GitHub 관리 규칙

GitHub에 올리는 파일:
- README.md
- requirements.txt
- setup.sh
- notebooks/*.ipynb
- src/*.py
- 실험 결과 요약 CSV

GitHub에 올리지 않는 파일:
- .env
- API key
- PDF 원본
- 대용량 데이터셋
- Chroma vectorstore
- 모델 체크포인트
- 캐시 파일

### 6.2 API Key 관리

API 키는 코드에 직접 쓰지 않는다.
환경변수 또는 .env 파일을 사용한다.

예시:
OPENAI_API_KEY=YOUR_API_KEY

주의:
.env 파일은 반드시 .gitignore에 포함해야 한다.

### 6.3 실험 기록 규칙

각 실험은 다음 정보를 남긴다.

- 실험명
- 날짜
- 문서 이름
- chunk_size
- chunk_overlap
- k
- embedding_model
- llm_model
- faithfulness 평균
- answer_relevancy 평균
- context_precision 평균
- context_recall 평균
- 관찰 내용

---

## 7. 예상 문제와 해결 방법

### 문제 1. ModuleNotFoundError: No module named 'ragas'

원인:
현재 Jupyter 커널에 ragas가 설치되어 있지 않다.

해결:
현재 커널의 Python에 직접 설치한다.
python -m pip install ragas==0.1.21

### 문제 2. ModuleNotFoundError: No module named 'datasets'

원인:
datasets 패키지가 현재 가상환경에 설치되어 있지 않다.

해결:
python -m pip install datasets

### 문제 3. langchain_community.chat_models.vertexai 오류

원인:
ragas와 langchain-community 버전이 맞지 않는다.

해결:
requirements.txt의 버전 조합을 사용한다.
ragas==0.1.21
langchain==0.1.20
langchain-community==0.0.38

### 문제 4. Chroma DB가 너무 커짐

원인:
chunk 실험을 반복하면서 vectorstore 폴더가 계속 증가한다.

해결:
실험 전 기존 vectorstore 폴더를 삭제하거나 실험별 폴더를 분리한다.

### 문제 5. 답변이 문서에 없는 내용을 말함

원인:
LLM이 context 밖의 지식을 사용했을 가능성이 있다.

해결:
프롬프트에 "제공된 문맥 안에서만 답변하라"는 조건을 추가한다.
faithfulness 점수를 확인한다.

---

## 8. 성능 해석 예시

실험 결과가 다음과 같다고 가정한다.

| 실험 | faithfulness | answer_relevancy | context_precision | context_recall |
|---|---:|---:|---:|---:|
| chunk_500 | 0.83 | 0.78 | 0.88 | 0.62 |
| chunk_1000 | 0.87 | 0.84 | 0.81 | 0.79 |
| chunk_1500 | 0.80 | 0.82 | 0.70 | 0.86 |

해석:
chunk_500은 검색된 문맥의 정밀도는 높지만 필요한 정보를 충분히 찾지 못했다.
chunk_1500은 필요한 정보는 많이 포함했지만 불필요한 정보가 섞여 정밀도가 낮았다.
chunk_1000은 faithfulness와 answer_relevancy가 가장 안정적이므로 기본 설정으로 적합하다.

---

## 9. 최종 보고서 구조

최종 보고서는 다음 순서로 작성한다.

1. 문제 정의
2. 데이터 설명
3. RAG 파이프라인 구조
4. 실험 설계
5. 실험 결과
6. 결과 해석
7. 한계
8. 개선 방향
9. 배포 방법
10. 회고

---

## 10. 추가 개선 아이디어

- 문서별 metadata filter 적용
- 질의 유형별 retriever 설정 변경
- BM25와 vector search를 결합한 hybrid search
- reranker 적용
- 답변에 source citation 추가
- Streamlit UI에 검색된 context 접기/펼치기 기능 추가
- 오류 기록 전용 RAG 봇으로 확장
- LangGraph 기반 agent로 확장
"""

path = Path("../data/rag_practice_complex_korean.txt")
path.parent.mkdir(parents=True, exist_ok=True)

path.write_text(content, encoding="utf-8")

print(f"생성 완료: {path}")
print(f"문자 수: {len(content):,}")

생성 완료: ..\data\rag_practice_complex_korean.txt
문자 수: 7,584


In [26]:
from langchain_community.document_loaders import TextLoader

TXT_PATH = "../data/rag_practice_complex_korean.txt"

loader = TextLoader(TXT_PATH, encoding="utf-8")
docs = loader.load()

print("문서 수:", len(docs))
print(docs[0].page_content[:500])

문서 수: 1
# AIFFEL 최종 프로젝트 운영 문서 v1.3
작성일: 2026-05-28
문서 목적: RAG 검색, 근거 기반 답변, 충돌 정보 탐지, 일정/정책/성능지표 질의응답 연습용 샘플 문서

---

## 1. 프로젝트 개요

본 문서는 AI 엔지니어링 학습자가 최종 프로젝트를 준비하면서 참고할 수 있는 운영 문서이다.
프로젝트의 기본 목표는 RAG(Retrieval-Augmented Generation)를 활용하여 개인 학습 자료, 오류 기록, 수업 노트, 프로젝트 산출물을 통합 검색하는 AI 학습 비서를 구현하는 것이다.

최종 산출물은 다음 네 가지 조건을 만족해야 한다.

1. 사용자가 업로드한 문서를 벡터 데이터베이스에 저장할 수 있어야 한다.
2. 질문이 들어오면 관련 문단을 검색하고, 검색된 근거를 함께 제시해야 한다.
3. RAGAS 또는 자체 평가 지표를 사용하여 답변 품질을 분석해야 한다.
4. FastAPI 또는 Streamlit을 이용해 간단한 데모 형태로 배포해


## 문서 로드 확인

In [27]:
from langchain_community.document_loaders import TextLoader

TXT_PATH = "../data/rag_practice_complex_korean.txt"

loader = TextLoader(TXT_PATH, encoding="utf-8")
docs = loader.load()

print("문서 수:", len(docs))
print(docs[0].page_content[:500])

문서 수: 1
# AIFFEL 최종 프로젝트 운영 문서 v1.3
작성일: 2026-05-28
문서 목적: RAG 검색, 근거 기반 답변, 충돌 정보 탐지, 일정/정책/성능지표 질의응답 연습용 샘플 문서

---

## 1. 프로젝트 개요

본 문서는 AI 엔지니어링 학습자가 최종 프로젝트를 준비하면서 참고할 수 있는 운영 문서이다.
프로젝트의 기본 목표는 RAG(Retrieval-Augmented Generation)를 활용하여 개인 학습 자료, 오류 기록, 수업 노트, 프로젝트 산출물을 통합 검색하는 AI 학습 비서를 구현하는 것이다.

최종 산출물은 다음 네 가지 조건을 만족해야 한다.

1. 사용자가 업로드한 문서를 벡터 데이터베이스에 저장할 수 있어야 한다.
2. 질문이 들어오면 관련 문단을 검색하고, 검색된 근거를 함께 제시해야 한다.
3. RAGAS 또는 자체 평가 지표를 사용하여 답변 품질을 분석해야 한다.
4. FastAPI 또는 Streamlit을 이용해 간단한 데모 형태로 배포해


In [28]:
eval_questions = [
    "이 프로젝트의 최종 목표는 무엇인가?",
    "chunk_size가 너무 작으면 어떤 문제가 생기는가?",
    "chunk_size가 너무 크면 어떤 문제가 생기는가?",
    "RAGAS에서 faithfulness는 무엇을 평가하는가?",
    "vectorstore 폴더를 GitHub에 올리지 않는 이유는 무엇인가?",
    "RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는 이유는 무엇인가?",
    "OpenAIEmbeddings를 사용하면 RunPod GPU를 많이 사용하는가?",
    "retriever의 k값을 높이면 항상 좋은가?",
    "배포형 데모에서 필요한 핵심 기능은 무엇인가?",
    "최종 발표에서 반드시 설명해야 할 항목은 무엇인가?",
]

ground_truths = [
    "이 프로젝트의 최종 목표는 RAG 기반 AI 학습 비서를 구현하고, 검색 품질과 답변 신뢰도를 평가 및 개선하는 것이다.",
    "chunk_size가 너무 작으면 문맥이 잘려 답변에 필요한 정보가 충분히 검색되지 않을 수 있다.",
    "chunk_size가 너무 크면 검색 결과에 불필요한 정보가 섞여 context_precision이 낮아질 수 있다.",
    "faithfulness는 답변이 검색된 context에 근거하여 사실적으로 작성되었는지를 평가한다.",
    "vectorstore는 용량이 크고 재생성 가능하기 때문에 GitHub에 올리지 않고, 생성 코드만 관리하는 것이 적절하다.",
    "최종 프로젝트의 모델과 데이터 규모가 확정되지 않았을 때는 저장공간을 먼저 고정하면 비용과 GPU 선택 제약이 생길 수 있기 때문이다.",
    "OpenAIEmbeddings는 OpenAI 서버에서 계산되므로 RunPod GPU 사용량은 크지 않다.",
    "아니다. k값을 높이면 recall은 좋아질 수 있지만 불필요한 문맥이 포함되어 precision이나 faithfulness가 낮아질 수 있다.",
    "문서 업로드, 질문 입력, 답변 출력, 근거 문단 출력, 간단한 사용 로그 저장 기능이 필요하다.",
    "문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.",
]

## RAG 실행 + 평가 함수 만들기

In [29]:
from datasets import Dataset

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma

from ragas import evaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

import pandas as pd
import os
import shutil

In [30]:
def run_rag_once(
    docs,
    questions,
    ground_truths,
    chunk_size=1000,
    chunk_overlap=100,
    k=3,
    experiment_name="exp"
):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    splits = splitter.split_documents(docs)
    print(f"[{experiment_name}] chunk 개수:", len(splits))

    persist_dir = f"../vectorstore/{experiment_name}"

    if os.path.exists(persist_dir):
        shutil.rmtree(persist_dir)

    embeddings = OpenAIEmbeddings()

    vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=embeddings
)

    retriever = vectorstore.as_retriever(
        search_kwargs={"k": k}
    )

    llm = ChatOpenAI(
        model="gpt-4o-mini",
        temperature=0
    )

    answers = []
    contexts = []

    for q in questions:
        retrieved_docs = retriever.invoke(q)
        context_texts = [doc.page_content for doc in retrieved_docs]

        context_joined = "\n\n".join(context_texts)

        prompt = f"""
너는 문서 기반 질의응답 AI다.

규칙:
1. 제공된 문맥에서 확인 가능한 내용을 바탕으로 답변해라.
2. 문맥에 질문과 직접 관련된 문장이 있으면 그 내용을 우선 사용해라.
3. 문맥에 '기준 답변:' 또는 명확한 설명이 있으면 그 내용을 자연스럽게 요약해라.
4. 정말 관련 정보가 전혀 없을 때만 '문서에서 확인되지 않습니다'라고 답해라.
5. 답변은 1~2문장으로 간결하게 작성해라.

[문맥]
{context_joined}

[질문]
{q}

[답변]
"""

        response = llm.invoke(prompt)
        answer = response.content

        answers.append(answer)
        contexts.append(context_texts)

    dataset = Dataset.from_dict({
        "question": questions,
        "answer": answers,
        "contexts": contexts,
        "ground_truth": ground_truths,
    })

    result = evaluate(
        dataset,
        metrics=[
            faithfulness,
            answer_relevancy,
            context_precision,
            context_recall,
        ]
    )

    result_df = result.to_pandas()

    result_df["experiment"] = experiment_name
    result_df["chunk_size"] = chunk_size
    result_df["chunk_overlap"] = chunk_overlap
    result_df["k"] = k

    return result_df

## chunk_size 실험 실행

In [31]:
exp_500 = run_rag_once(
    docs=docs,
    questions=eval_questions,
    ground_truths=ground_truths,
    chunk_size=500,
    chunk_overlap=50,
    k=3,
    experiment_name="chunk_500"
)

[chunk_500] chunk 개수: 19


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

In [32]:
exp_1000 = run_rag_once(
    docs=docs,
    questions=eval_questions,
    ground_truths=ground_truths,
    chunk_size=1000,
    chunk_overlap=100,
    k=3,
    experiment_name="chunk_1000"
)

exp_1500 = run_rag_once(
    docs=docs,
    questions=eval_questions,
    ground_truths=ground_truths,
    chunk_size=1500,
    chunk_overlap=150,
    k=3,
    experiment_name="chunk_1500"
)

[chunk_1000] chunk 개수: 9


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

[chunk_1500] chunk 개수: 6


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

## 결과 합치기

In [34]:
all_results = pd.concat(
    [exp_500, exp_1000, exp_1500],
    ignore_index=True
)

all_results

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall,experiment,chunk_size,chunk_overlap,k
0,이 프로젝트의 최종 목표는 무엇인가?,[최종 산출물은 다음 네 가지 조건을 만족해야 한다.\n\n1. 사용자가 업로드한 ...,이 프로젝트의 최종 목표는 검색 품질과 답변 신뢰도를 실험하고 개선하는 것이다.,"이 프로젝트의 최종 목표는 RAG 기반 AI 학습 비서를 구현하고, 검색 품질과 답...",1.0,0.887002,1.0,1.0,chunk_500,500,50,3
1,chunk_size가 너무 작으면 어떤 문제가 생기는가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,chunk_size가 너무 작으면 문맥이 잘려 답변에 필요한 정보가 충분히 검색되지...,chunk_size가 너무 작으면 문맥이 잘려 답변에 필요한 정보가 충분히 검색되지...,0.5,0.873142,1.0,1.0,chunk_500,500,50,3
2,chunk_size가 너무 크면 어떤 문제가 생기는가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,chunk_size가 너무 크면 검색 결과에 불필요한 정보가 섞여 context_p...,chunk_size가 너무 크면 검색 결과에 불필요한 정보가 섞여 context_p...,1.0,0.834041,1.0,1.0,chunk_500,500,50,3
3,RAGAS에서 faithfulness는 무엇을 평가하는가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,faithfulness는 답변이 검색된 context에 근거하여 사실적으로 작성되었...,faithfulness는 답변이 검색된 context에 근거하여 사실적으로 작성되었...,1.0,0.842116,1.0,1.0,chunk_500,500,50,3
4,vectorstore 폴더를 GitHub에 올리지 않는 이유는 무엇인가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,vectorstore는 용량이 크고 재생성 가능하기 때문에 GitHub에 올리지 않...,vectorstore는 용량이 크고 재생성 가능하기 때문에 GitHub에 올리지 않...,1.0,0.813627,1.0,1.0,chunk_500,500,50,3
5,RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는 이유는 ...,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,최종 프로젝트의 모델과 데이터 규모가 확정되지 않았을 때는 저장공간을 먼저 고정하면...,최종 프로젝트의 모델과 데이터 규모가 확정되지 않았을 때는 저장공간을 먼저 고정하면...,1.0,0.760723,1.0,1.0,chunk_500,500,50,3
6,OpenAIEmbeddings를 사용하면 RunPod GPU를 많이 사용하는가?,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,OpenAIEmbeddings는 OpenAI 서버에서 계산되므로 RunPod GPU...,OpenAIEmbeddings는 OpenAI 서버에서 계산되므로 RunPod GPU...,1.0,0.891709,1.0,1.0,chunk_500,500,50,3
7,retriever의 k값을 높이면 항상 좋은가?,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,아니다. k값을 높이면 recall은 좋아질 수 있지만 불필요한 문맥이 포함되어 p...,아니다. k값을 높이면 recall은 좋아질 수 있지만 불필요한 문맥이 포함되어 p...,0.8,0.793593,1.0,1.0,chunk_500,500,50,3
8,배포형 데모에서 필요한 핵심 기능은 무엇인가?,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,"배포형 데모에서 필요한 핵심 기능은 문서 업로드, 질문 입력, 답변 출력, 근거 문...","문서 업로드, 질문 입력, 답변 출력, 근거 문단 출력, 간단한 사용 로그 저장 기...",1.0,0.826471,1.0,1.0,chunk_500,500,50,3
9,최종 발표에서 반드시 설명해야 할 항목은 무엇인가?,[Q10. 최종 발표에서 반드시 설명해야 할 항목은 무엇인가?\n기준 답변: 문제 ...,"문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.","문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.",1.0,0.826626,1.0,1.0,chunk_500,500,50,3


## 평균 점수 보기

In [35]:
score_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

summary = all_results.groupby(
    ["experiment", "chunk_size", "chunk_overlap", "k"]
)[score_cols].mean().reset_index()

summary

,experiment,chunk_size,chunk_overlap,k,faithfulness,answer_relevancy,context_precision,context_recall
0,chunk_1000,1000,100,3,0.93,0.682709,1.0,1.0
1,chunk_1500,1500,150,3,0.93,0.768631,1.0,1.0
2,chunk_500,500,50,3,0.93,0.834905,1.0,1.0


## 결과 저장

In [36]:
os.makedirs("../results", exist_ok=True)

all_results.to_csv("../results/ragas_all_results.csv", index=False, encoding="utf-8-sig")
summary.to_csv("../results/ragas_summary.csv", index=False, encoding="utf-8-sig")

print("결과 저장 완료")

결과 저장 완료


In [37]:
from pathlib import Path

print("현재 작업 폴더:", Path.cwd())
print(".env 존재 여부:", Path(".env").exists())
print("../.env 존재 여부:", Path("../.env").exists())

현재 작업 폴더: c:\AI_study\rag-eval-practice\notebooks
.env 존재 여부: False
../.env 존재 여부: True


In [38]:
from dotenv import load_dotenv
import os

load_dotenv("../.env")

print("API 키 로드 여부:", os.environ.get("OPENAI_API_KEY") is not None)

API 키 로드 여부: True


## chunk_size 500 / 1000 / 1500 실험

In [39]:
print("docs:", type(docs), len(docs))
print("eval_questions:", len(eval_questions))
print("ground_truths:", len(ground_truths))
print("run_rag_once:", callable(run_rag_once))

docs: <class 'list'> 1
eval_questions: 10
ground_truths: 10
run_rag_once: True


In [40]:
exp_500 = run_rag_once(
    docs=docs,
    questions=eval_questions,
    ground_truths=ground_truths,
    chunk_size=500,
    chunk_overlap=50,
    k=3,
    experiment_name="chunk_500"
)

exp_500

[chunk_500] chunk 개수: 19


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

,user_input,retrieved_contexts,response,reference,faithfulness,answer_relevancy,context_precision,context_recall,experiment,chunk_size,chunk_overlap,k
0,이 프로젝트의 최종 목표는 무엇인가?,[최종 산출물은 다음 네 가지 조건을 만족해야 한다.\n\n1. 사용자가 업로드한 ...,이 프로젝트의 최종 목표는 검색 품질과 답변 신뢰도를 실험하고 개선하는 것이다.,"이 프로젝트의 최종 목표는 RAG 기반 AI 학습 비서를 구현하고, 검색 품질과 답...",1.0,0.889581,1.0,1.0,chunk_500,500,50,3
1,chunk_size가 너무 작으면 어떤 문제가 생기는가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,chunk_size가 너무 작으면 문맥이 잘려 답변에 필요한 정보가 충분히 검색되지...,chunk_size가 너무 작으면 문맥이 잘려 답변에 필요한 정보가 충분히 검색되지...,1.0,0.873142,1.0,1.0,chunk_500,500,50,3
2,chunk_size가 너무 크면 어떤 문제가 생기는가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,chunk_size가 너무 크면 검색 결과에 불필요한 정보가 섞여 context_p...,chunk_size가 너무 크면 검색 결과에 불필요한 정보가 섞여 context_p...,1.0,0.834041,1.0,1.0,chunk_500,500,50,3
3,RAGAS에서 faithfulness는 무엇을 평가하는가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,faithfulness는 답변이 검색된 context에 근거하여 사실적으로 작성되었...,faithfulness는 답변이 검색된 context에 근거하여 사실적으로 작성되었...,1.0,0.840535,1.0,1.0,chunk_500,500,50,3
4,vectorstore 폴더를 GitHub에 올리지 않는 이유는 무엇인가?,[Q2. chunk_size가 너무 작으면 어떤 문제가 생기는가?\n기준 답변: c...,vectorstore는 용량이 크고 재생성 가능하기 때문에 GitHub에 올리지 않...,vectorstore는 용량이 크고 재생성 가능하기 때문에 GitHub에 올리지 않...,1.0,0.813627,1.0,1.0,chunk_500,500,50,3
5,RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는 이유는 ...,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,최종 프로젝트의 모델과 데이터 규모가 확정되지 않았을 때는 저장공간을 먼저 고정하면...,최종 프로젝트의 모델과 데이터 규모가 확정되지 않았을 때는 저장공간을 먼저 고정하면...,1.0,0.809329,1.0,1.0,chunk_500,500,50,3
6,OpenAIEmbeddings를 사용하면 RunPod GPU를 많이 사용하는가?,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,OpenAIEmbeddings는 OpenAI 서버에서 계산되므로 RunPod GPU...,OpenAIEmbeddings는 OpenAI 서버에서 계산되므로 RunPod GPU...,1.0,0.885602,1.0,1.0,chunk_500,500,50,3
7,retriever의 k값을 높이면 항상 좋은가?,[### 4.2 Retriever k값 실험\n\n동일한 chunk 설정에서 k값만...,아니다. k값을 높이면 recall은 좋아질 수 있지만 불필요한 문맥이 포함되어 p...,아니다. k값을 높이면 recall은 좋아질 수 있지만 불필요한 문맥이 포함되어 p...,0.8,0.793593,1.0,1.0,chunk_500,500,50,3
8,배포형 데모에서 필요한 핵심 기능은 무엇인가?,[Q6. RunPod를 사용할 때 Network Volume을 나중에 고려해도 되는...,"배포형 데모에서 필요한 핵심 기능은 문서 업로드, 질문 입력, 답변 출력, 근거 문...","문서 업로드, 질문 입력, 답변 출력, 근거 문단 출력, 간단한 사용 로그 저장 기...",1.0,0.826471,1.0,1.0,chunk_500,500,50,3
9,최종 발표에서 반드시 설명해야 할 항목은 무엇인가?,[Q10. 최종 발표에서 반드시 설명해야 할 항목은 무엇인가?\n기준 답변: 문제 ...,"문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.","문제 정의, 시스템 구조, 실험 결과, 한계, 개선 방향을 설명해야 한다.",1.0,0.826626,1.0,1.0,chunk_500,500,50,3


In [41]:
score_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

exp_500[score_cols].mean()

faithfulness         0.980000
answer_relevancy     0.839255
context_precision    1.000000
context_recall       1.000000
dtype: float64

In [42]:
exp_500[
    (exp_500["faithfulness"] < 0.8) |
    (exp_500["answer_relevancy"] < 0.7)
][[
    "user_input",
    "response",
    "reference",
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]]

,user_input,response,reference,faithfulness,answer_relevancy,context_precision,context_recall


In [43]:
bad_rows = exp_500[
    (exp_500["faithfulness"] < 0.8) |
    (exp_500["answer_relevancy"] < 0.7)
]

for idx, row in bad_rows.iterrows():
    print("=" * 80)
    print("INDEX:", idx)
    print("질문:", row["user_input"])
    print("\n답변:", row["response"])
    print("\n기준 답변:", row["reference"])
    print("\n검색 문맥:")
    for i, ctx in enumerate(row["retrieved_contexts"], 1):
        print(f"\n--- context {i} ---")
        print(ctx[:1000])

In [44]:
exp_1000 = run_rag_once(
    docs=docs,
    questions=eval_questions,
    ground_truths=ground_truths,
    chunk_size=1000,
    chunk_overlap=100,
    k=3,
    experiment_name="chunk_1000"
)

exp_1500 = run_rag_once(
    docs=docs,
    questions=eval_questions,
    ground_truths=ground_truths,
    chunk_size=1500,
    chunk_overlap=150,
    k=3,
    experiment_name="chunk_1500"
)

[chunk_1000] chunk 개수: 9


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

[chunk_1500] chunk 개수: 6


Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

## 평균 점수 summary 생성

In [45]:
all_results = pd.concat(
    [exp_500, exp_1000, exp_1500],
    ignore_index=True
)

score_cols = [
    "faithfulness",
    "answer_relevancy",
    "context_precision",
    "context_recall"
]

summary = all_results.groupby(
    ["experiment", "chunk_size", "chunk_overlap", "k"]
)[score_cols].mean().reset_index()

summary

,experiment,chunk_size,chunk_overlap,k,faithfulness,answer_relevancy,context_precision,context_recall
0,chunk_1000,1000,100,3,0.93,0.840745,1.0,1.0
1,chunk_1500,1500,150,3,0.93,0.758249,1.0,1.0
2,chunk_500,500,50,3,0.98,0.839255,1.0,1.0


In [46]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 1000)

summary

,experiment,chunk_size,chunk_overlap,k,faithfulness,answer_relevancy,context_precision,context_recall
0,chunk_1000,1000,100,3,0.93,0.840745,1.0,1.0
1,chunk_1500,1500,150,3,0.93,0.758249,1.0,1.0
2,chunk_500,500,50,3,0.98,0.839255,1.0,1.0


In [47]:
summary[summary["experiment"] == "chunk_500"]

,experiment,chunk_size,chunk_overlap,k,faithfulness,answer_relevancy,context_precision,context_recall
2,chunk_500,500,50,3,0.98,0.839255,1.0,1.0


## 결과 CSV 저장


In [48]:
import os

os.makedirs("../results", exist_ok=True)

all_results.to_csv(
    "../results/ragas_all_results.csv",
    index=False,
    encoding="utf-8-sig"
)

summary.to_csv(
    "../results/ragas_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

print("결과 저장 완료")

결과 저장 완료
